# Dynamic Tag-Anchored IFRS S1/S2 Evidence Mapper — Corrected

This notebook replaces the previous record-level tag mapper with a stricter two-stage pipeline:

1. **Evidence tags and requirement text retrieve candidate schema fields dynamically.**
2. **A semantic mapper validates the exact IFRS clause and selects only schema IDs that directly support it.**
3. Selected schema IDs are resolved to **exact payload field paths**, grouped by their original record.
4. Multi-year records use composite identities, preventing 2022/2023 records from overwriting 2024.
5. Conditional, report-design, not-applicable and unavailable requirements are separated from disclosure coverage.
6. Cross-section evidence is available through a bank-wide schema catalogue.
7. Coverage is only asserted when all material clause components are supported.

No IFRS requirement ID, bank code, exact payload filename, or requirement-to-field mapping is hardcoded.


In [ ]:
# ============================================================
# CELL 1 — Imports, configuration and dynamic input discovery
# ============================================================

import os
import re
import json
import math
import time
import hashlib
import urllib.request
import urllib.error
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(items, **kwargs):
        return items


def locate_notebook_dir(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if candidate.name.lower() == "notebooks":
            return candidate
        nested = candidate / "notebooks"
        if nested.exists() and nested.is_dir():
            return nested.resolve()
    return start


CURRENT_DIR = Path.cwd().resolve()
NOTEBOOK_DIR = locate_notebook_dir(CURRENT_DIR)
GEN_DATA_DIR = NOTEBOOK_DIR / "gen_data"

PAYLOAD_DIR = Path(
    os.getenv("PAYLOAD_DIR", str(GEN_DATA_DIR / "payloads_risk"))
).resolve()

REQUIREMENTS_DIR = Path(
    os.getenv(
        "IFRS_REQUIREMENTS_DIR",
        str(
            GEN_DATA_DIR
            / "IFRS"
            / "ifrs_requirements_kb_outputs_final"
            / "section_by_section_requirements"
            / "json"
        ),
    )
).resolve()

GENERATION_OUTPUT_DIR = Path(
    os.getenv(
        "GENERATION_OUTPUT_DIR",
        str(GEN_DATA_DIR / "generated_reports" / "agentic_ifrs_report"),
    )
).resolve()

MAPPING_OUTPUT_DIR = Path(
    os.getenv(
        "MAPPING_OUTPUT_DIR",
        str(GENERATION_OUTPUT_DIR / "00_requirement_mapping"),
    )
).resolve()

BANK_CODE = os.getenv("BANK_CODE", "").strip() or None
SECTIONS_TO_RUN = {
    value.strip()
    for value in os.getenv("MAPPING_SECTIONS", "").split(",")
    if value.strip()
}

USE_LLM_MAPPING = os.getenv("USE_LLM_MAPPING", "true").lower() in {
    "1", "true", "yes", "y"
}
STRICT_LLM_MAPPING = os.getenv("STRICT_LLM_MAPPING", "true").lower() in {
    "1", "true", "yes", "y"
}

LLM_BATCH_SIZE = int(os.getenv("MAPPING_LLM_BATCH_SIZE", "8"))
TOP_SCHEMA_CANDIDATES = int(os.getenv("TOP_SCHEMA_CANDIDATES", "80"))
MAX_SCHEMA_CANDIDATES_PER_BATCH = int(
    os.getenv("MAX_SCHEMA_CANDIDATES_PER_BATCH", "420")
)
MIN_CANDIDATES_PER_REQUIREMENT = int(
    os.getenv("MIN_CANDIDATES_PER_REQUIREMENT", "30")
)
MAX_SCHEMAS_PER_REQUIREMENT = int(
    os.getenv("MAX_SCHEMAS_PER_REQUIREMENT", "15")
)
MAX_CONCRETE_PATHS_PER_SCHEMA = int(
    os.getenv("MAX_CONCRETE_PATHS_PER_SCHEMA", "30")
)
MAX_TOTAL_EVIDENCE_PATHS = int(
    os.getenv("MAX_TOTAL_EVIDENCE_PATHS", "60")
)
REPRESENTATIVE_RECORD_LIMIT = int(
    os.getenv("REPRESENTATIVE_RECORD_LIMIT", "3")
)

MAPPING_PROMPT_VERSION = "tag_schema_mapper_v3_component_coverage_2026_07_28"

SECTION_ALIASES = {
    "general_requirements": "general_requirements",
    "general_requirement": "general_requirements",
    "governance": "governance",
    "strategy": "strategy",
    "risk_management": "risk_management",
    "risk": "risk_management",
    "metrics_targets": "metrics_and_targets",
    "metrics_and_targets": "metrics_and_targets",
    "metrics_target": "metrics_and_targets",
}


def normalize_section_key(value: str) -> str:
    value = re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")
    return SECTION_ALIASES.get(value, value)


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def infer_payload_identity(path: Path) -> Tuple[Optional[str], Optional[str]]:
    stem = re.sub(r"\s*\(\d+\)$", "", path.stem).strip()
    section = None
    bank = None

    for alias in sorted(SECTION_ALIASES, key=len, reverse=True):
        suffix = "_" + alias
        if stem.lower().endswith(suffix):
            section = normalize_section_key(alias)
            prefix = stem[: -len(suffix)]
            bank = re.sub(r"^payload_", "", prefix, flags=re.IGNORECASE)
            break

    try:
        payload = load_json(path)
        metadata = payload.get("metadata", {}) if isinstance(payload, dict) else {}
        bank_object = payload.get("bank", {}) if isinstance(payload, dict) else {}
        bank = metadata.get("bank_id") or bank_object.get("bank_id") or bank
    except Exception:
        pass

    return (str(bank) if bank else None, section)


def discover_input_pairs() -> Tuple[str, Dict[str, Dict[str, Path]]]:
    if not PAYLOAD_DIR.exists():
        raise FileNotFoundError(f"Payload directory not found: {PAYLOAD_DIR}")
    if not REQUIREMENTS_DIR.exists():
        raise FileNotFoundError(
            f"Requirements directory not found: {REQUIREMENTS_DIR}"
        )

    requirement_files = {}
    for path in sorted(REQUIREMENTS_DIR.glob("*.json")):
        try:
            document = load_json(path)
        except Exception:
            continue

        if not isinstance(document, dict) or "standards" not in document:
            continue

        section = normalize_section_key(
            document.get("section_key")
            or path.stem.replace("_requirements", "")
        )
        if section in SECTION_ALIASES.values():
            requirement_files[section] = path

    payload_candidates = []
    for path in sorted(PAYLOAD_DIR.glob("payload_BANK01_*.json")):
        bank, section = infer_payload_identity(path)
        if bank and section:
            payload_candidates.append((bank, section, path))

    available_banks = sorted({bank for bank, _, _ in payload_candidates})
    resolved_bank = BANK_CODE

    if resolved_bank is None:
        if len(available_banks) == 1:
            resolved_bank = available_banks[0]
        elif not available_banks:
            raise FileNotFoundError(
                f"No valid payload files found in {PAYLOAD_DIR}"
            )
        else:
            raise ValueError(
                "Multiple banks were found. Set BANK_CODE. "
                f"Available banks: {available_banks}"
            )

    payload_files = {
        section: path
        for bank, section, path in payload_candidates
        if bank == resolved_bank
    }

    sections = sorted(set(payload_files) & set(requirement_files))
    if SECTIONS_TO_RUN:
        requested = {normalize_section_key(x) for x in SECTIONS_TO_RUN}
        sections = [section for section in sections if section in requested]

    if not sections:
        raise ValueError(
            f"No matching payload/requirement pairs for bank {resolved_bank}."
        )

    pairs = {
        section: {
            "payload": payload_files[section],
            "requirements": requirement_files[section],
        }
        for section in sections
    }

    print("Bank:", resolved_bank)
    print("Payload directory:", PAYLOAD_DIR)
    print("Requirements directory:", REQUIREMENTS_DIR)
    print("Output directory:", MAPPING_OUTPUT_DIR)
    print("Mode:", "tag-guided schema + LLM" if USE_LLM_MAPPING else "schema-only")

    for section, files in pairs.items():
        print(f"\n{section}")
        print("  payload:", files["payload"])
        print("  requirements:", files["requirements"])

    return resolved_bank, pairs


RESOLVED_BANK_CODE, RESOLVED_FILES = discover_input_pairs()

PAYLOADS = {
    section: load_json(files["payload"])
    for section, files in RESOLVED_FILES.items()
}
REQUIREMENTS_DOCUMENTS = {
    section: load_json(files["requirements"])
    for section, files in RESOLVED_FILES.items()
}

REPORTING_YEAR = next(
    (
        int(payload.get("metadata", {}).get("reporting_year"))
        for payload in PAYLOADS.values()
        if payload.get("metadata", {}).get("reporting_year") is not None
    ),
    None,
)

print("\nReporting year:", REPORTING_YEAR)


In [ ]:
# ============================================================
# CELL 2 — Bank-wide schema catalogue and tag-guided retrieval
# ============================================================

EMPTY_STRINGS = {
    "", "none", "null", "nan", "n/a", "na", "not available"
}

TOKEN_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "has", "have", "how", "if", "in", "information", "is", "it", "its",
    "of", "on", "or", "shall", "that", "the", "their", "this", "to",
    "used", "uses", "using", "was", "were", "whether", "which", "with",
    "entity", "disclose", "disclosed", "disclosure", "disclosures",
    "metric", "metrics", "target", "targets", "requirement", "requirements",
}

TOKEN_ALIASES = {
    "pct": "percentage",
    "percent": "percentage",
    "percentage": "percentage",
    "meur": "amount",
    "eur": "amount",
    "usd": "amount",
    "gbp": "amount",
    "tco2e": "emission",
    "co2e": "emission",
    "ghg": "emission",
}


def is_empty(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if isinstance(value, str) and value.strip().lower() in EMPTY_STRINGS:
        return True
    if isinstance(value, (list, dict)) and not value:
        return True
    return False


def preview(value: Any, limit: int = 220) -> str:
    text = (
        json.dumps(value, ensure_ascii=False)
        if isinstance(value, (dict, list))
        else str(value)
    )
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit] + ("…" if len(text) > limit else "")


def tokenize(value: Any) -> List[str]:
    text = str(value or "")
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    normalized = []
    for token in tokens:
        if token in TOKEN_STOPWORDS or len(token) <= 1:
            continue
        if token.endswith("ies") and len(token) > 4:
            token = token[:-3] + "y"
        elif token.endswith("s") and len(token) > 4 and not token.endswith("ss"):
            token = token[:-1]
        token = TOKEN_ALIASES.get(token, token)
        normalized.append(token)
    return normalized


def generalized_path(path: str) -> str:
    return re.sub(r"\[\d+\]", "[]", path)


def detect_context_fields(record: Dict[str, Any]) -> Dict[str, Any]:
    context = {}
    for key, value in record.items():
        if isinstance(value, (dict, list)) or is_empty(value):
            continue

        normalized = str(key).lower()
        if (
            normalized == "reporting_year"
            or normalized.endswith("_id")
            or normalized.endswith("_year")
            or normalized.endswith("_date")
            or normalized.endswith("_name")
            or normalized.endswith("_type")
            or normalized.endswith("_category")
            or normalized.endswith("_status")
            or normalized.endswith("_scope")
            or normalized.endswith("_horizon")
        ):
            context[key] = value

    return context


def record_key_for(
    section: str,
    record_path: str,
    context: Dict[str, Any],
) -> str:
    identity_context = {
        key: value
        for key, value in context.items()
        if (
            key == "reporting_year"
            or key.endswith("_id")
            or key.endswith("_year")
            or key.endswith("_date")
            or key.endswith("_name")
        )
    }
    raw = {
        "section": section,
        "record_path": record_path,
        "identity_context": identity_context,
    }
    digest = hashlib.sha256(
        json.dumps(raw, sort_keys=True, ensure_ascii=False).encode("utf-8")
    ).hexdigest()[:16]
    return f"{section}::{generalized_path(record_path) or '<root>'}::{digest}"


def flatten_payload(
    section: str,
    value: Any,
    prefix: str = "",
    inherited_context: Optional[Dict[str, Any]] = None,
) -> List[Dict[str, Any]]:
    inherited_context = dict(inherited_context or {})
    rows = []

    if isinstance(value, dict):
        local_context = {
            **inherited_context,
            **detect_context_fields(value),
        }

        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)

            if isinstance(child, (dict, list)):
                rows.extend(
                    flatten_payload(section, child, path, local_context)
                )
            else:
                parent_path = prefix
                qualified_path = f"{section}::{path}"
                qualified_parent = f"{section}::{parent_path or '<root>'}"
                rows.append({
                    "source_section": section,
                    "path": path,
                    "qualified_path": qualified_path,
                    "schema_path": generalized_path(path),
                    "qualified_schema_path": (
                        f"{section}::{generalized_path(path)}"
                    ),
                    "parent_path": parent_path,
                    "qualified_parent_path": qualified_parent,
                    "schema_parent_path": generalized_path(parent_path),
                    "root": path.split(".", 1)[0].split("[", 1)[0],
                    "leaf": str(key),
                    "value": child,
                    "value_type": type(child).__name__,
                    "context": local_context,
                    "record_key": record_key_for(
                        section,
                        parent_path,
                        local_context,
                    ),
                })

    elif isinstance(value, list):
        for index, child in enumerate(value):
            rows.extend(
                flatten_payload(
                    section,
                    child,
                    f"{prefix}[{index}]",
                    inherited_context,
                )
            )

    return rows


def unique_preserving_order(values: List[Any]) -> List[Any]:
    seen = set()
    result = []

    for value in values:
        marker = json.dumps(value, sort_keys=True, ensure_ascii=False)
        if marker in seen:
            continue
        seen.add(marker)
        result.append(value)

    return result


def build_bank_schema_catalog(
    payloads: Dict[str, Dict[str, Any]],
) -> Tuple[
    List[Dict[str, Any]],
    Dict[str, Dict[str, Any]],
    Dict[str, List[Dict[str, Any]]],
    Dict[str, Any],
]:
    flat_rows = []
    for section, payload in payloads.items():
        flat_rows.extend(
            row
            for row in flatten_payload(section, payload)
            if not is_empty(row["value"])
        )

    grouped = defaultdict(list)
    path_index = {}

    for row in flat_rows:
        grouped[row["qualified_schema_path"]].append(row)
        path_index[row["qualified_path"]] = row["value"]

    schema_catalog = []
    schema_by_id = {}
    rows_by_schema_id = {}

    for qualified_schema_path in sorted(grouped):
        rows = grouped[qualified_schema_path]
        schema_id = "F" + hashlib.sha256(
            qualified_schema_path.encode("utf-8")
        ).hexdigest()[:12].upper()

        sample_values = unique_preserving_order([
            preview(row["value"], 160) for row in rows
        ])[:4]

        context_summary = defaultdict(list)
        for row in rows:
            for key, value in row["context"].items():
                context_summary[key].append(value)

        context_summary = {
            key: unique_preserving_order(values)[:5]
            for key, values in context_summary.items()
        }

        token_source = " ".join([
            rows[0]["source_section"],
            rows[0]["schema_path"],
            rows[0]["root"],
            rows[0]["leaf"],
            " ".join(map(str, sample_values)),
            " ".join(context_summary.keys()),
            " ".join(
                str(v)
                for values in context_summary.values()
                for v in values[:2]
            ),
        ])

        schema_item = {
            "schema_id": schema_id,
            "source_section": rows[0]["source_section"],
            "schema_path": rows[0]["schema_path"],
            "qualified_schema_path": qualified_schema_path,
            "root": rows[0]["root"],
            "leaf": rows[0]["leaf"],
            "value_types": sorted({
                row["value_type"] for row in rows
            }),
            "record_count": len({
                row["record_key"] for row in rows
            }),
            "value_count": len(rows),
            "sample_values": sample_values,
            "context_summary": context_summary,
            "_tokens": set(tokenize(token_source)),
        }

        schema_catalog.append(schema_item)
        schema_by_id[schema_id] = schema_item
        rows_by_schema_id[schema_id] = rows

    return schema_catalog, schema_by_id, rows_by_schema_id, path_index


def iter_requirements(document: Dict[str, Any]) -> List[Dict[str, Any]]:
    requirements = []
    for block in document.get("standards", {}).values():
        requirements.extend(block.get("requirements", []))
    return requirements


def requirement_text_blob(requirement: Dict[str, Any]) -> str:
    return str(requirement.get("requirement_text") or "")


def requirement_auxiliary_tokens(
    requirement: Dict[str, Any],
) -> Tuple[set, set]:
    heading_tokens = set(tokenize(" ".join([
        str(requirement.get("official_section_heading") or ""),
        str(requirement.get("nearest_pdf_heading") or ""),
    ])))
    tag_tokens = set(
        tokenize(" ".join(requirement.get("evidence_tags") or []))
    )
    return heading_tokens, tag_tokens


def build_schema_idf(
    schema_catalog: List[Dict[str, Any]],
) -> Dict[str, float]:
    document_frequency = Counter()
    total = len(schema_catalog)
    for item in schema_catalog:
        document_frequency.update(item["_tokens"])

    return {
        token: math.log((total + 1) / (count + 1)) + 1
        for token, count in document_frequency.items()
    }


def candidate_score(
    requirement: Dict[str, Any],
    section: str,
    schema_item: Dict[str, Any],
    idf: Dict[str, float],
) -> float:
    requirement_tokens = set(tokenize(requirement_text_blob(requirement)))
    heading_tokens, tag_tokens = requirement_auxiliary_tokens(requirement)
    schema_tokens = schema_item["_tokens"]

    content_overlap = requirement_tokens & schema_tokens
    score = 2.5 * sum(idf.get(token, 1.0) for token in content_overlap)

    # Evidence tags are retrieval hints only. They receive a small boost so
    # generic tags such as metrics/targets cannot drown out clause wording.
    tag_overlap = tag_tokens & schema_tokens
    score += 0.6 * sum(idf.get(token, 1.0) for token in tag_overlap)

    heading_overlap = heading_tokens & schema_tokens
    score += 0.25 * sum(idf.get(token, 1.0) for token in heading_overlap)

    leaf_tokens = set(tokenize(schema_item["leaf"]))
    score += 3.0 * sum(
        idf.get(token, 1.0)
        for token in requirement_tokens & leaf_tokens
    )

    path_tokens = set(tokenize(schema_item["schema_path"]))
    score += 1.5 * sum(
        idf.get(token, 1.0)
        for token in requirement_tokens & path_tokens
    )

    root_tokens = set(tokenize(schema_item["root"]))
    score += 1.0 * sum(
        idf.get(token, 1.0)
        for token in requirement_tokens & root_tokens
    )

    if schema_item["source_section"] == section:
        score += 0.5

    if schema_item["root"] == "metadata":
        score -= 0.5

    return score


def candidate_schema_for_requirement(
    requirement: Dict[str, Any],
    section: str,
    schema_catalog: List[Dict[str, Any]],
    idf: Dict[str, float],
) -> List[Dict[str, Any]]:
    scored = [
        (
            candidate_score(
                requirement,
                section,
                schema_item,
                idf,
            ),
            schema_item,
        )
        for schema_item in schema_catalog
    ]

    scored.sort(
        key=lambda item: (
            item[0],
            item[1]["source_section"] == section,
            -item[1]["record_count"],
        ),
        reverse=True,
    )

    positive = [
        {**item, "retrieval_score": round(score, 4)}
        for score, item in scored
        if score > 0
    ]

    if len(positive) < min(20, TOP_SCHEMA_CANDIDATES):
        fallback = [
            {**item, "retrieval_score": round(score, 4)}
            for score, item in scored
            if item["source_section"] == section
        ]
        by_id = {item["schema_id"]: item for item in positive}
        for item in fallback:
            by_id.setdefault(item["schema_id"], item)
        positive = list(by_id.values())
        positive.sort(
            key=lambda item: item["retrieval_score"],
            reverse=True,
        )

    return positive[:TOP_SCHEMA_CANDIDATES]


def compact_schema_for_prompt(
    schema_items: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    return [
        {
            "schema_id": row["schema_id"],
            "source_section": row["source_section"],
            "path": row["schema_path"],
            "types": row["value_types"],
            "samples": row["sample_values"],
            "context": row["context_summary"],
            "record_count": row["record_count"],
            "retrieval_score": row.get("retrieval_score"),
        }
        for row in schema_items
    ]


(
    BANK_SCHEMA_CATALOG,
    SCHEMA_BY_ID,
    ROWS_BY_SCHEMA_ID,
    QUALIFIED_PATH_INDEX,
) = build_bank_schema_catalog(PAYLOADS)

SCHEMA_IDF = build_schema_idf(BANK_SCHEMA_CATALOG)

print("Bank-wide generalized schema fields:", len(BANK_SCHEMA_CATALOG))
print("Exact non-empty payload fields:", len(QUALIFIED_PATH_INDEX))


In [ ]:
# ============================================================
# CELL 3 — Clause-level semantic mapper, validation and cache
# ============================================================

AZURE_OPENAI_API_KEY = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("OPENAI_API_KEY")
)
AZURE_OPENAI_FAST_DEPLOYMENT_URL = os.getenv(
    "AZURE_OPENAI_FAST_DEPLOYMENT_URL"
)
AZURE_OPENAI_GPT52_DEPLOYMENT_URL = os.getenv(
    "AZURE_OPENAI_GPT52_DEPLOYMENT_URL"
)
MAPPING_LLM_URL = (
    AZURE_OPENAI_FAST_DEPLOYMENT_URL
    or AZURE_OPENAI_GPT52_DEPLOYMENT_URL
)

VALID_MAPPING_STATUSES = {
    "covered",
    "partially_covered",
    "not_available_in_payload",
    "conditional_not_triggered",
    "handled_by_report_design",
    "not_applicable_to_entity_scope",
}

VALID_RECORD_SCOPES = {
    "current_period",
    "comparative_periods",
    "all_records",
    "non_period_specific",
    "representative_records",
}


def parse_json_object(raw: str) -> Dict[str, Any]:
    raw = str(raw).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end = raw.rfind("}")
        if start < 0 or end <= start:
            raise
        return json.loads(raw[start:end + 1])


def standalone_azure_json(
    messages: List[Dict[str, str]],
    max_tokens: int = 12000,
    retries: int = 5,
) -> Dict[str, Any]:
    if not AZURE_OPENAI_API_KEY or not MAPPING_LLM_URL:
        raise ValueError(
            "LLM mapping is enabled but Azure/OpenAI configuration is missing. "
            "Set AZURE_OPENAI_API_KEY and AZURE_OPENAI_FAST_DEPLOYMENT_URL, "
            "or set USE_LLM_MAPPING=false."
        )

    last_error = None

    request_variants = [
        ("max_completion_tokens", True),
        ("max_completion_tokens", False),
        ("max_tokens", True),
        ("max_tokens", False),
    ]

    for token_field, use_response_format in request_variants:
        request_body = {
            "messages": messages,
            token_field: max_tokens,
        }
        if use_response_format:
            request_body["response_format"] = {"type": "json_object"}

        for attempt in range(1, retries + 1):
            request = urllib.request.Request(
                MAPPING_LLM_URL,
                data=json.dumps(request_body).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": AZURE_OPENAI_API_KEY,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(
                    request,
                    timeout=240,
                ) as response:
                    result = json.loads(
                        response.read().decode("utf-8")
                    )

                content = result["choices"][0]["message"]["content"]
                return parse_json_object(content)

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"Mapping LLM HTTP {exc.code}: {body[:1200]}"
                )

                if exc.code == 400:
                    # Try the next parameter combination. Azure deployments
                    # differ on token-field and response-format support.
                    break
                if exc.code == 429 or exc.code >= 500:
                    time.sleep(min(30, attempt * 4))
                    continue
                raise last_error

            except Exception as exc:
                last_error = exc
                if attempt < retries:
                    time.sleep(min(30, attempt * 4))
                    continue
                break

    raise RuntimeError(f"Mapping LLM failed: {last_error}")


def call_mapping_llm(
    messages: List[Dict[str, str]],
) -> Dict[str, Any]:
    existing_helper = globals().get("azure_chat_json")

    if callable(existing_helper):
        try:
            result = existing_helper(
                messages=messages,
                model_tier="fast",
                temperature=0,
                max_tokens=12000,
                request_label="tag-guided IFRS schema mapper",
            )
        except TypeError:
            result = existing_helper(messages)

        if isinstance(result, str):
            return parse_json_object(result)
        return result

    return standalone_azure_json(messages)


SCHEMA_MAPPER_SYSTEM_PROMPT = """
You map IFRS S1 and IFRS S2 requirements to a supplied bank payload schema.

Each requirement has a candidate_schema_ids list. You may select ONLY those
candidate schema IDs. The evidence tags are retrieval anchors, not proof of
coverage.

For every requirement:
1. Decompose the exact clause into its material atomic disclosure components.
2. Mark each component supported only when one or more candidate fields directly
   support that component.
3. Do not infer a process, policy, control, responsibility, methodology,
   explanation, trade-off, consistency statement or decision from a metric,
   flag, outcome, meeting occurrence or related subject matter.
4. Do not use a numeric match when semantic meaning, unit, period or record
   identity differs.
5. Use covered only when every material component is supported.
6. Use partially_covered only when at least one component is supported and at
   least one component is unsupported.
7. Use not_available_in_payload when none of the material components is directly
   supported.
8. Use conditional_not_triggered only when the clause itself is conditional and
   selected status fields directly prove that the trigger did not occur.
9. Use not_applicable_to_entity_scope only when selected status fields directly
   prove that the activity is outside the entity's scope.
10. Use handled_by_report_design only for presentation, duplication,
    cross-reference, placement, labelling or report-assembly controls.
11. Use the narrowest evidence possible. Do not select whole families of related
    metrics. Never select more than 15 schema fields for one requirement.
12. Choose record_scope:
    - current_period: current reporting year only;
    - comparative_periods: current year plus the immediately preceding available year;
    - all_records: all matching records are necessary;
    - non_period_specific: policy or metadata fields without a reporting year;
    - representative_records: at most a small sample is needed to demonstrate
      a process or record structure.
13. Return JSON only.

Output:
{
  "mappings": [
    {
      "requirement_id": "...",
      "mapping_status": "covered|partially_covered|not_available_in_payload|conditional_not_triggered|handled_by_report_design|not_applicable_to_entity_scope",
      "components": [
        {
          "component": "atomic part of the clause",
          "supported": true,
          "schema_ids": ["F..."],
          "record_scope": "current_period",
          "support_reason": "why these fields directly support this component"
        }
      ],
      "status_evidence": [
        {
          "schema_id": "F...",
          "record_scope": "non_period_specific",
          "support_reason": "why this field proves the trigger is absent or scope is inapplicable"
        }
      ],
      "missing_information": ["specific missing disclosure information"],
      "rationale": "overall mapping decision",
      "confidence": 0.0
    }
  ]
}
""".strip()


def requirement_for_prompt(
    requirement: Dict[str, Any],
    candidate_ids: List[str],
) -> Dict[str, Any]:
    text = str(requirement.get("requirement_text") or "").lower()
    return {
        "requirement_id": requirement["requirement_id"],
        "standard": requirement.get("standard"),
        "paragraph_id": requirement.get("paragraph_id"),
        "clause_path": requirement.get("clause_path"),
        "requirement_text": requirement.get("requirement_text"),
        "official_section_heading": requirement.get(
            "official_section_heading"
        ),
        "evidence_tags": requirement.get("evidence_tags") or [],
        "mandatory": requirement.get("mandatory"),
        "candidate_schema_ids": candidate_ids,
        "linguistic_hints": {
            "contains_conditional_language": bool(
                re.search(r"\b(if|when|unless|in the absence of)\b", text)
            ),
            "contains_presentation_language": bool(
                re.search(
                    r"\b(duplication|cross-reference|location|label|identify clearly|placement)\b",
                    text,
                )
            ),
        },
    }


def validate_evidence_item(
    requirement_id: str,
    item: Dict[str, Any],
    allowed_ids: set,
) -> Dict[str, Any]:
    schema_id = item.get("schema_id")
    record_scope = item.get("record_scope")

    if schema_id not in allowed_ids:
        raise ValueError(
            f"{requirement_id} selected schema outside its candidate set: "
            f"{schema_id}"
        )
    if schema_id not in SCHEMA_BY_ID:
        raise ValueError(
            f"{requirement_id} selected unknown schema ID: {schema_id}"
        )
    if record_scope not in VALID_RECORD_SCOPES:
        raise ValueError(
            f"{requirement_id} returned invalid record scope: {record_scope}"
        )

    return {
        "schema_id": schema_id,
        "source_section": SCHEMA_BY_ID[schema_id]["source_section"],
        "schema_path": SCHEMA_BY_ID[schema_id]["schema_path"],
        "record_scope": record_scope,
        "support_reason": str(item.get("support_reason", "")).strip(),
    }


def validate_llm_mapping(
    requirement: Dict[str, Any],
    raw: Dict[str, Any],
    candidate_ids: List[str],
) -> Dict[str, Any]:
    requirement_id = requirement["requirement_id"]
    allowed_ids = set(candidate_ids)

    if raw.get("requirement_id") != requirement_id:
        raise ValueError(
            f"Requirement mismatch: expected {requirement_id}, "
            f"received {raw.get('requirement_id')}"
        )

    status = raw.get("mapping_status")
    if status not in VALID_MAPPING_STATUSES:
        raise ValueError(
            f"Invalid status for {requirement_id}: {status}"
        )

    components = raw.get("components") or []
    if not isinstance(components, list):
        raise ValueError(
            f"{requirement_id} returned invalid components."
        )

    validated_components = []
    disclosure_evidence = []
    supported_count = 0
    unsupported_count = 0

    for component in components:
        if not isinstance(component, dict):
            raise ValueError(
                f"{requirement_id} returned a non-object component."
            )

        supported = bool(component.get("supported"))
        schema_ids = component.get("schema_ids") or []
        if not isinstance(schema_ids, list):
            raise ValueError(
                f"{requirement_id} returned invalid schema_ids."
            )

        record_scope = component.get("record_scope")
        support_reason = str(
            component.get("support_reason", "")
        ).strip()

        component_evidence = []
        for schema_id in schema_ids:
            component_evidence.append(
                validate_evidence_item(
                    requirement_id,
                    {
                        "schema_id": schema_id,
                        "record_scope": record_scope,
                        "support_reason": support_reason,
                    },
                    allowed_ids,
                )
            )

        if supported and not component_evidence:
            raise ValueError(
                f"{requirement_id} marks a component supported without evidence."
            )
        if not supported and component_evidence:
            raise ValueError(
                f"{requirement_id} attaches evidence to an unsupported component."
            )

        if supported:
            supported_count += 1
            disclosure_evidence.extend(component_evidence)
        else:
            unsupported_count += 1

        validated_components.append({
            "component": str(component.get("component", "")).strip(),
            "supported": supported,
            "schema_evidence": component_evidence,
            "support_reason": support_reason,
        })

    status_evidence = [
        validate_evidence_item(requirement_id, item, allowed_ids)
        for item in (raw.get("status_evidence") or [])
    ]

    if len({
        (item["schema_id"], item["record_scope"])
        for item in disclosure_evidence
    }) > MAX_SCHEMAS_PER_REQUIREMENT:
        raise ValueError(
            f"{requirement_id} selected more than "
            f"{MAX_SCHEMAS_PER_REQUIREMENT} schema fields."
        )

    missing_information = raw.get("missing_information") or []
    if not isinstance(missing_information, list):
        missing_information = [str(missing_information)]
    missing_information = [
        str(item).strip()
        for item in missing_information
        if str(item).strip()
    ]

    if status == "covered":
        if not components or unsupported_count or not disclosure_evidence:
            raise ValueError(
                f"{requirement_id} returned covered without complete component support."
            )
        if missing_information:
            raise ValueError(
                f"{requirement_id} returned covered with missing information."
            )
        if status_evidence:
            raise ValueError(
                f"{requirement_id} returned covered with status evidence."
            )

    elif status == "partially_covered":
        if not supported_count or not unsupported_count or not disclosure_evidence:
            raise ValueError(
                f"{requirement_id} returned partially_covered without mixed support."
            )
        if not missing_information:
            missing_information = [
                component["component"]
                for component in validated_components
                if not component["supported"]
                and component["component"]
            ]
        if status_evidence:
            raise ValueError(
                f"{requirement_id} returned partial coverage with status evidence."
            )

    elif status == "not_available_in_payload":
        if supported_count or disclosure_evidence or status_evidence:
            raise ValueError(
                f"{requirement_id} returned unavailable with evidence."
            )
        if not components:
            raise ValueError(
                f"{requirement_id} returned unavailable without clause components."
            )

    elif status in {
        "conditional_not_triggered",
        "not_applicable_to_entity_scope",
    }:
        if disclosure_evidence:
            raise ValueError(
                f"{requirement_id} returned {status} with disclosure evidence."
            )
        if not status_evidence:
            raise ValueError(
                f"{requirement_id} returned {status} without status evidence."
            )

    elif status == "handled_by_report_design":
        if disclosure_evidence or status_evidence:
            raise ValueError(
                f"{requirement_id} returned report-design status with evidence."
            )

    deduplicated_evidence = []
    seen = set()
    for item in disclosure_evidence:
        key = (item["schema_id"], item["record_scope"])
        if key in seen:
            continue
        seen.add(key)
        deduplicated_evidence.append(item)

    deduplicated_status_evidence = []
    seen = set()
    for item in status_evidence:
        key = (item["schema_id"], item["record_scope"])
        if key in seen:
            continue
        seen.add(key)
        deduplicated_status_evidence.append(item)

    try:
        confidence = float(raw.get("confidence", 0.0))
    except (TypeError, ValueError):
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    return {
        "requirement_id": requirement_id,
        "mapping_status": status,
        "coverage_components": validated_components,
        "schema_evidence": deduplicated_evidence,
        "status_schema_evidence": deduplicated_status_evidence,
        "missing_information": missing_information,
        "mapping_rationale": str(raw.get("rationale", "")).strip(),
        "mapping_confidence": confidence,
        "mapping_method": "tag_guided_schema_plus_llm",
        "prompt_version": MAPPING_PROMPT_VERSION,
    }


def cache_key(
    requirement: Dict[str, Any],
    candidate_schema: List[Dict[str, Any]],
) -> str:
    content = {
        "prompt_version": MAPPING_PROMPT_VERSION,
        "requirement": requirement_for_prompt(
            requirement,
            [item["schema_id"] for item in candidate_schema],
        ),
        "schema": compact_schema_for_prompt(candidate_schema),
    }
    return hashlib.sha256(
        json.dumps(
            content,
            sort_keys=True,
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()


def cache_path(
    cache_dir: Path,
    requirement: Dict[str, Any],
    candidate_schema: List[Dict[str, Any]],
) -> Path:
    return cache_dir / (
        requirement["requirement_id"]
        + "__"
        + cache_key(requirement, candidate_schema)[:16]
        + ".json"
    )


def load_cached_result(
    cache_dir: Path,
    requirement: Dict[str, Any],
    candidate_schema: List[Dict[str, Any]],
) -> Optional[Dict[str, Any]]:
    path = cache_path(cache_dir, requirement, candidate_schema)
    if not path.exists():
        return None
    return load_json(path)


def save_cached_result(
    cache_dir: Path,
    requirement: Dict[str, Any],
    candidate_schema: List[Dict[str, Any]],
    mapping: Dict[str, Any],
) -> None:
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_path(cache_dir, requirement, candidate_schema)
    path.write_text(
        json.dumps(mapping, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )


def map_requirement_batch(
    batch_items: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    schema_union = {}
    for item in batch_items:
        for schema_item in item["candidate_schema"]:
            schema_union[schema_item["schema_id"]] = schema_item

    # Guarantee each requirement its strongest candidates before filling
    # the remaining shared prompt budget. This prevents one broad requirement
    # from crowding another requirement's cross-section evidence out of a batch.
    guaranteed_ids = set()
    for item in batch_items:
        guaranteed_ids.update(
            schema["schema_id"]
            for schema in item["candidate_schema"][
                :MIN_CANDIDATES_PER_REQUIREMENT
            ]
        )

    globally_ranked = sorted(
        schema_union.values(),
        key=lambda item: item.get("retrieval_score", 0),
        reverse=True,
    )

    schema_union_rows = [
        schema_union[schema_id]
        for schema_id in guaranteed_ids
        if schema_id in schema_union
    ]
    included_ids = {
        item["schema_id"] for item in schema_union_rows
    }

    for schema_item in globally_ranked:
        if len(schema_union_rows) >= MAX_SCHEMA_CANDIDATES_PER_BATCH:
            break
        if schema_item["schema_id"] in included_ids:
            continue
        schema_union_rows.append(schema_item)
        included_ids.add(schema_item["schema_id"])

    schema_union_rows.sort(
        key=lambda item: item.get("retrieval_score", 0),
        reverse=True,
    )
    allowed_union_ids = {
        item["schema_id"] for item in schema_union_rows
    }

    prompt_requirements = []
    effective_candidates = {}

    for item in batch_items:
        candidate_ids = [
            schema["schema_id"]
            for schema in item["candidate_schema"]
            if schema["schema_id"] in allowed_union_ids
        ]
        effective_candidates[item["requirement"]["requirement_id"]] = candidate_ids
        prompt_requirements.append(
            requirement_for_prompt(
                item["requirement"],
                candidate_ids,
            )
        )

    response = call_mapping_llm([
        {"role": "system", "content": SCHEMA_MAPPER_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": json.dumps(
                {
                    "requirements": prompt_requirements,
                    "payload_schema": compact_schema_for_prompt(
                        schema_union_rows
                    ),
                },
                ensure_ascii=False,
            ),
        },
    ])

    returned = {
        item.get("requirement_id"): item
        for item in response.get("mappings", [])
        if isinstance(item, dict)
    }

    results = []
    for item in batch_items:
        requirement = item["requirement"]
        requirement_id = requirement["requirement_id"]
        if requirement_id not in returned:
            raise ValueError(
                f"LLM omitted requirement {requirement_id}"
            )
        results.append(
            validate_llm_mapping(
                requirement,
                returned[requirement_id],
                effective_candidates[requirement_id],
            )
        )

    return results


def schema_only_result(
    requirement: Dict[str, Any],
) -> Dict[str, Any]:
    return {
        "requirement_id": requirement["requirement_id"],
        "mapping_status": "needs_semantic_validation",
        "coverage_components": [],
        "schema_evidence": [],
        "status_schema_evidence": [],
        "missing_information": [],
        "mapping_rationale": (
            "Candidate schema retrieval completed, but semantic mapping was "
            "disabled. No coverage status has been asserted."
        ),
        "mapping_confidence": 0.0,
        "mapping_method": "tag_guided_schema_only",
        "prompt_version": MAPPING_PROMPT_VERSION,
    }


In [ ]:
# ============================================================
# CELL 4 — Exact path resolution, period control and data gaps
# ============================================================

def row_reporting_year(row: Dict[str, Any]) -> Optional[int]:
    value = row.get("context", {}).get("reporting_year")
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def select_concrete_rows(
    rows: List[Dict[str, Any]],
    record_scope: str,
    reporting_year: Optional[int],
) -> List[Dict[str, Any]]:
    if record_scope == "current_period":
        selected = [
            row for row in rows
            if row_reporting_year(row) == reporting_year
        ]
        if selected:
            return selected

        return [
            row for row in rows
            if row_reporting_year(row) is None
        ]

    if record_scope == "comparative_periods":
        period_rows = [
            row for row in rows
            if row_reporting_year(row) is not None
            and reporting_year is not None
            and row_reporting_year(row) <= reporting_year
        ]
        available_years = sorted({
            row_reporting_year(row)
            for row in period_rows
        })
        selected_years = []
        if reporting_year in available_years:
            selected_years.append(reporting_year)
        prior_years = [
            year for year in available_years
            if year < reporting_year
        ]
        if prior_years:
            selected_years.append(max(prior_years))

        selected = [
            row for row in period_rows
            if row_reporting_year(row) in selected_years
        ]
        return selected

    if record_scope == "non_period_specific":
        return [
            row for row in rows
            if row_reporting_year(row) is None
        ]

    if record_scope == "representative_records":
        current = [
            row for row in rows
            if row_reporting_year(row) == reporting_year
        ]
        source = current or [
            row for row in rows
            if row_reporting_year(row) is None
        ] or rows

        selected = []
        seen_records = set()
        for row in source:
            if row["record_key"] in seen_records:
                continue
            seen_records.add(row["record_key"])
            selected.append(row)
            if len(selected) >= REPRESENTATIVE_RECORD_LIMIT:
                break
        return selected

    return rows


def resolve_schema_rows(
    requirement_id: str,
    evidence_items: List[Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], List[str]]:
    concrete_rows = []
    warnings = []

    for evidence in evidence_items:
        schema_id = evidence["schema_id"]
        selected = select_concrete_rows(
            ROWS_BY_SCHEMA_ID[schema_id],
            evidence["record_scope"],
            REPORTING_YEAR,
        )

        if not selected:
            raise ValueError(
                f"{requirement_id} selected {schema_id} with scope "
                f"{evidence['record_scope']}, but no records resolved."
            )

        if len(selected) > MAX_CONCRETE_PATHS_PER_SCHEMA:
            warnings.append(
                f"{schema_id} resolved to {len(selected)} paths; retained "
                f"the first {MAX_CONCRETE_PATHS_PER_SCHEMA}."
            )
            selected = selected[:MAX_CONCRETE_PATHS_PER_SCHEMA]

        for row in selected:
            concrete_rows.append({
                "schema_id": schema_id,
                "source_section": row["source_section"],
                "schema_path": evidence["schema_path"],
                "record_scope": evidence["record_scope"],
                "support_reason": evidence["support_reason"],
                "path": row["path"],
                "qualified_path": row["qualified_path"],
                "parent_path": row["parent_path"],
                "qualified_parent_path": row["qualified_parent_path"],
                "record_key": row["record_key"],
                "value": row["value"],
                "value_type": row["value_type"],
                "context": row["context"],
            })

    deduplicated = []
    seen = set()
    for row in concrete_rows:
        key = (row["schema_id"], row["qualified_path"])
        if key in seen:
            continue
        seen.add(key)
        deduplicated.append(row)

    if len(deduplicated) > MAX_TOTAL_EVIDENCE_PATHS:
        warnings.append(
            f"Evidence bundle had {len(deduplicated)} exact paths; retained "
            f"the first {MAX_TOTAL_EVIDENCE_PATHS}."
        )
        deduplicated = deduplicated[:MAX_TOTAL_EVIDENCE_PATHS]

    return deduplicated, warnings


def group_resolved_evidence(
    rows: List[Dict[str, Any]],
) -> List[Dict[str, Any]]:
    groups = defaultdict(list)
    for row in rows:
        groups[row["record_key"]].append(row)

    result = []
    for record_key, fields in sorted(groups.items()):
        first = fields[0]
        result.append({
            "record_key": record_key,
            "source_section": first["source_section"],
            "record_path": first["parent_path"],
            "qualified_record_path": first["qualified_parent_path"],
            "record_context": first["context"],
            "fields": fields,
        })
    return result


def all_declared_data_gaps(
    payloads: Dict[str, Dict[str, Any]],
) -> List[Dict[str, Any]]:
    gaps = []
    seen = set()
    for section, payload in payloads.items():
        for gap in payload.get("metadata", {}).get("data_gaps", []) or []:
            marker = json.dumps(gap, sort_keys=True, ensure_ascii=False)
            if marker in seen:
                continue
            seen.add(marker)
            gaps.append({"source_section": section, **gap})
    return gaps


DECLARED_DATA_GAPS = all_declared_data_gaps(PAYLOADS)


def match_declared_data_gaps(
    requirement: Dict[str, Any],
    selected_rows: List[Dict[str, Any]],
    mapping_status: str,
) -> List[Dict[str, Any]]:
    if mapping_status not in {
        "partially_covered",
        "not_available_in_payload",
        "conditional_not_triggered",
    }:
        return []

    requirement_tokens = set(tokenize(requirement_text_blob(requirement)))
    selected_path_tokens = set(
        tokenize(" ".join(row["qualified_path"] for row in selected_rows))
    )

    matches = []
    for gap in DECLARED_DATA_GAPS:
        field_tokens = set(tokenize(str(gap.get("field") or "")))
        detail_tokens = set(tokenize(" ".join([
            str(gap.get("reason") or ""),
            str(gap.get("instruction") or ""),
        ])))

        field_overlap = (
            requirement_tokens | selected_path_tokens
        ) & field_tokens
        detail_overlap = requirement_tokens & detail_tokens

        # A gap must match the affected field itself. Generic words appearing
        # only in a reason/instruction are insufficient.
        if not field_overlap:
            continue
        if not selected_rows and len(field_overlap | detail_overlap) < 2:
            continue

        affected_years = gap.get("affected_years") or []
        if affected_years and REPORTING_YEAR not in affected_years:
            continue

        matches.append({
            **gap,
            "matched_tokens": sorted(field_overlap | detail_overlap),
        })

    matches.sort(
        key=lambda gap: len(gap["matched_tokens"]),
        reverse=True,
    )
    return matches[:5]


def resolve_mapping_evidence(
    section: str,
    requirement: Dict[str, Any],
    mapping: Dict[str, Any],
) -> Dict[str, Any]:
    disclosure_rows, disclosure_warnings = resolve_schema_rows(
        mapping["requirement_id"],
        mapping.get("schema_evidence") or [],
    )
    status_rows, status_warnings = resolve_schema_rows(
        mapping["requirement_id"],
        mapping.get("status_schema_evidence") or [],
    )

    source_sections = sorted({
        row["source_section"]
        for row in disclosure_rows + status_rows
    })

    return {
        **mapping,
        "candidate_section": section,
        "cross_section": any(
            source_section != section
            for source_section in source_sections
        ),
        "selected_source_sections": source_sections,
        "selected_evidence_paths": [
            row["qualified_path"] for row in disclosure_rows
        ],
        "selected_evidence": disclosure_rows,
        "evidence_groups": group_resolved_evidence(disclosure_rows),
        "status_evidence_paths": [
            row["qualified_path"] for row in status_rows
        ],
        "status_evidence": status_rows,
        "status_evidence_groups": group_resolved_evidence(status_rows),
        "declared_data_gaps": match_declared_data_gaps(
            requirement,
            disclosure_rows + status_rows,
            mapping["mapping_status"],
        ),
        "resolution_warnings": disclosure_warnings + status_warnings,
    }


In [ ]:
# ============================================================
# CELL 5 — Mapping orchestration with batch-to-single retry
# ============================================================

def map_one_uncached(
    item: Dict[str, Any],
) -> Dict[str, Any]:
    try:
        return map_requirement_batch([item])[0]
    except Exception:
        if STRICT_LLM_MAPPING:
            raise
        return schema_only_result(item["requirement"])


def map_section(
    section: str,
    requirements_document: Dict[str, Any],
) -> Tuple[List[Dict[str, Any]], Dict[str, List[Dict[str, Any]]]]:
    requirements = iter_requirements(requirements_document)
    cache_dir = MAPPING_OUTPUT_DIR / "cache" / section

    candidate_by_requirement = {
        requirement["requirement_id"]: candidate_schema_for_requirement(
            requirement,
            section,
            BANK_SCHEMA_CATALOG,
            SCHEMA_IDF,
        )
        for requirement in requirements
    }

    mapping_by_requirement = {}
    uncached_items = []

    for requirement in requirements:
        requirement_id = requirement["requirement_id"]
        candidate_schema = candidate_by_requirement[requirement_id]

        if not USE_LLM_MAPPING:
            mapping_by_requirement[requirement_id] = schema_only_result(
                requirement
            )
            continue

        cached = load_cached_result(
            cache_dir,
            requirement,
            candidate_schema,
        )
        if cached is not None:
            mapping_by_requirement[requirement_id] = cached
        else:
            uncached_items.append({
                "requirement": requirement,
                "candidate_schema": candidate_schema,
            })

    if USE_LLM_MAPPING:
        for start in tqdm(
            range(0, len(uncached_items), LLM_BATCH_SIZE),
            desc=f"Clause mapping — {section}",
        ):
            batch = uncached_items[start:start + LLM_BATCH_SIZE]

            try:
                batch_results = map_requirement_batch(batch)
            except Exception as batch_error:
                print(
                    f"Batch validation failed in {section}; retrying "
                    f"{len(batch)} requirement(s) individually: {batch_error}"
                )
                batch_results = [
                    map_one_uncached(item)
                    for item in batch
                ]

            for item, mapping in zip(batch, batch_results):
                requirement = item["requirement"]
                requirement_id = requirement["requirement_id"]

                if mapping["mapping_method"] == "tag_guided_schema_plus_llm":
                    save_cached_result(
                        cache_dir,
                        requirement,
                        item["candidate_schema"],
                        mapping,
                    )

                mapping_by_requirement[requirement_id] = mapping

    rows = []
    for requirement in requirements:
        requirement_id = requirement["requirement_id"]
        mapping = resolve_mapping_evidence(
            section,
            requirement,
            mapping_by_requirement[requirement_id],
        )

        rows.append({
            "section_key": section,
            "requirement_id": requirement_id,
            "standard": requirement.get("standard"),
            "paragraph_id": requirement.get("paragraph_id"),
            "clause_path": requirement.get("clause_path"),
            "official_section_heading": requirement.get(
                "official_section_heading"
            ),
            "requirement_text": requirement.get("requirement_text"),
            "evidence_tags": requirement.get("evidence_tags") or [],
            "mandatory": requirement.get("mandatory"),
            "candidate_schema_count": len(
                candidate_by_requirement[requirement_id]
            ),
            "top_candidate_schema": compact_schema_for_prompt(
                candidate_by_requirement[requirement_id][:10]
            ),
            **mapping,
        })

    return rows, candidate_by_requirement


MAPPINGS_BY_SECTION = {}
CANDIDATES_BY_SECTION = {}

for section, document in REQUIREMENTS_DOCUMENTS.items():
    (
        MAPPINGS_BY_SECTION[section],
        CANDIDATES_BY_SECTION[section],
    ) = map_section(section, document)

ALL_MAPPINGS = [
    row
    for section_rows in MAPPINGS_BY_SECTION.values()
    for row in section_rows
]

print("Sections mapped:", list(MAPPINGS_BY_SECTION))
print("Total requirements:", len(ALL_MAPPINGS))
print(
    "Status counts:",
    dict(Counter(row["mapping_status"] for row in ALL_MAPPINGS)),
)


In [ ]:
# ============================================================
# CELL 6 — Validation, composite evidence store and exports
# ============================================================

VALID_OUTPUT_STATUSES = VALID_MAPPING_STATUSES | {
    "needs_semantic_validation"
}


def validate_mapping_outputs() -> Dict[str, Any]:
    requirement_ids = []
    source_requirement_ids = []

    for section, document in REQUIREMENTS_DOCUMENTS.items():
        source_requirement_ids.extend(
            requirement["requirement_id"]
            for requirement in iter_requirements(document)
        )
        requirement_ids.extend(
            row["requirement_id"]
            for row in MAPPINGS_BY_SECTION[section]
        )

    invalid_paths = []
    value_mismatches = []
    wrong_period_evidence = []
    oversized_bundles = []
    truncated_evidence = []
    status_errors = []
    duplicate_record_keys = []

    for row in ALL_MAPPINGS:
        if row["mapping_status"] not in VALID_OUTPUT_STATUSES:
            status_errors.append({
                "requirement_id": row["requirement_id"],
                "error": "invalid_status",
            })

        all_evidence = (
            row["selected_evidence"]
            + row.get("status_evidence", [])
        )

        if len(row["selected_evidence_paths"]) > MAX_TOTAL_EVIDENCE_PATHS:
            oversized_bundles.append(row["requirement_id"])

        if any(
            "retained the first" in warning
            for warning in row.get("resolution_warnings", [])
        ):
            truncated_evidence.append({
                "requirement_id": row["requirement_id"],
                "warnings": row.get("resolution_warnings", []),
            })

        for evidence in all_evidence:
            qualified_path = evidence["qualified_path"]
            if qualified_path not in QUALIFIED_PATH_INDEX:
                invalid_paths.append({
                    "requirement_id": row["requirement_id"],
                    "path": qualified_path,
                })
                continue

            if QUALIFIED_PATH_INDEX[qualified_path] != evidence["value"]:
                value_mismatches.append({
                    "requirement_id": row["requirement_id"],
                    "path": qualified_path,
                })

            year = row_reporting_year(evidence)
            scope = evidence["record_scope"]
            if scope == "current_period" and year not in {
                REPORTING_YEAR, None
            }:
                wrong_period_evidence.append({
                    "requirement_id": row["requirement_id"],
                    "path": qualified_path,
                    "year": year,
                    "scope": scope,
                })
            if scope == "non_period_specific" and year is not None:
                wrong_period_evidence.append({
                    "requirement_id": row["requirement_id"],
                    "path": qualified_path,
                    "year": year,
                    "scope": scope,
                })

        record_keys = [
            group["record_key"]
            for group in (
                row["evidence_groups"]
                + row.get("status_evidence_groups", [])
            )
        ]
        if len(record_keys) != len(set(record_keys)):
            duplicate_record_keys.append(row["requirement_id"])

        status = row["mapping_status"]
        disclosure_paths = row["selected_evidence_paths"]
        status_paths = row.get("status_evidence_paths", [])
        components = row.get("coverage_components", [])
        supported = [c for c in components if c.get("supported")]
        unsupported = [c for c in components if not c.get("supported")]

        if status == "covered" and (
            not disclosure_paths
            or unsupported
            or row["missing_information"]
        ):
            status_errors.append({
                "requirement_id": row["requirement_id"],
                "error": "covered_inconsistent",
            })
        elif status == "partially_covered" and (
            not disclosure_paths
            or not supported
            or not unsupported
        ):
            status_errors.append({
                "requirement_id": row["requirement_id"],
                "error": "partial_inconsistent",
            })
        elif status in {
            "conditional_not_triggered",
            "not_applicable_to_entity_scope",
        } and (
            disclosure_paths or not status_paths
        ):
            status_errors.append({
                "requirement_id": row["requirement_id"],
                "error": "status_justification_inconsistent",
            })
        elif status in {
            "not_available_in_payload",
            "handled_by_report_design",
            "needs_semantic_validation",
        } and (
            disclosure_paths or status_paths
        ):
            status_errors.append({
                "requirement_id": row["requirement_id"],
                "error": "no_evidence_status_has_paths",
            })

    tests = {
        "requirement_count_matches": (
            len(requirement_ids)
            == len(source_requirement_ids)
            == len(set(source_requirement_ids))
        ),
        "all_requirement_ids_mapped": (
            set(requirement_ids) == set(source_requirement_ids)
        ),
        "requirement_ids_unique": (
            len(requirement_ids) == len(set(requirement_ids))
        ),
        "all_exact_paths_resolve": not invalid_paths,
        "all_resolved_values_match_payload": not value_mismatches,
        "period_scopes_are_respected": not wrong_period_evidence,
        "evidence_bundles_are_bounded": not oversized_bundles,
        "no_evidence_was_silently_truncated": not truncated_evidence,
        "status_and_evidence_are_consistent": not status_errors,
        "record_keys_are_unique_per_mapping": not duplicate_record_keys,
        "every_disclosure_evidence_item_has_exact_path": all(
            evidence.get("qualified_path")
            for row in ALL_MAPPINGS
            for evidence in row["selected_evidence"]
        ),
    }

    return {
        "result": "PASS" if all(tests.values()) else "FAIL",
        "tests": tests,
        "invalid_paths": invalid_paths,
        "value_mismatches": value_mismatches,
        "wrong_period_evidence": wrong_period_evidence,
        "oversized_bundles": oversized_bundles,
        "truncated_evidence": truncated_evidence,
        "status_errors": status_errors,
        "duplicate_record_keys": duplicate_record_keys,
    }


VALIDATION_REPORT = validate_mapping_outputs()


def build_evidence_store(
    mappings: List[Dict[str, Any]],
) -> Dict[str, Any]:
    store = {}

    for row in mappings:
        for role, groups in (
            ("disclosure_support", row["evidence_groups"]),
            ("status_justification", row.get("status_evidence_groups", [])),
        ):
            for group in groups:
                record_key = group["record_key"]
                record = store.setdefault(record_key, {
                    "record_key": record_key,
                    "source_section": group["source_section"],
                    "record_path": group["record_path"],
                    "qualified_record_path": group["qualified_record_path"],
                    "record_context": group["record_context"],
                    "fields": {},
                    "used_by": [],
                })

                for field in group["fields"]:
                    record["fields"][field["qualified_path"]] = field["value"]

                usage = {
                    "requirement_id": row["requirement_id"],
                    "section_key": row["section_key"],
                    "role": role,
                }
                if usage not in record["used_by"]:
                    record["used_by"].append(usage)

    return store


EVIDENCE_STORE = build_evidence_store(ALL_MAPPINGS)

MAPPING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mapping_path = (
    MAPPING_OUTPUT_DIR
    / f"mapping_{RESOLVED_BANK_CODE}.json"
)
evidence_store_path = (
    MAPPING_OUTPUT_DIR
    / f"evidence_store_{RESOLVED_BANK_CODE}.json"
)
schema_path = (
    MAPPING_OUTPUT_DIR
    / f"schema_catalog_{RESOLVED_BANK_CODE}.json"
)
validation_path = (
    MAPPING_OUTPUT_DIR
    / f"mapping_validation_{RESOLVED_BANK_CODE}.json"
)

mapping_path.write_text(
    json.dumps(ALL_MAPPINGS, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
evidence_store_path.write_text(
    json.dumps(EVIDENCE_STORE, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
schema_path.write_text(
    json.dumps(
        [
            {
                key: value
                for key, value in item.items()
                if key != "_tokens"
            }
            for item in BANK_SCHEMA_CATALOG
        ],
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)
validation_path.write_text(
    json.dumps(
        VALIDATION_REPORT,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

for section, rows in MAPPINGS_BY_SECTION.items():
    (
        MAPPING_OUTPUT_DIR
        / f"mapping_{RESOLVED_BANK_CODE}_{section}.json"
    ).write_text(
        json.dumps(rows, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

SUMMARY = {
    "bank_code": RESOLVED_BANK_CODE,
    "reporting_year": REPORTING_YEAR,
    "mapping_method": (
        "tag_guided_schema_plus_llm"
        if USE_LLM_MAPPING
        else "tag_guided_schema_only"
    ),
    "total_requirements": len(ALL_MAPPINGS),
    "status_counts": dict(
        Counter(row["mapping_status"] for row in ALL_MAPPINGS)
    ),
    "cross_section_mappings": sum(
        bool(row["cross_section"])
        for row in ALL_MAPPINGS
    ),
    "exact_evidence_paths": sum(
        len(row["selected_evidence_paths"])
        for row in ALL_MAPPINGS
    ),
    "composite_evidence_records": len(EVIDENCE_STORE),
    "requirements_with_declared_gaps": sum(
        bool(row["declared_data_gaps"])
        for row in ALL_MAPPINGS
    ),
    "validation_result": VALIDATION_REPORT["result"],
    "output_files": {
        "mapping": str(mapping_path),
        "evidence_store": str(evidence_store_path),
        "schema_catalog": str(schema_path),
        "validation": str(validation_path),
    },
}

summary_path = (
    MAPPING_OUTPUT_DIR
    / f"mapping_summary_{RESOLVED_BANK_CODE}.json"
)
summary_path.write_text(
    json.dumps(SUMMARY, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(SUMMARY, indent=2))
print("\nValidation:", VALIDATION_REPORT["result"])
if VALIDATION_REPORT["result"] != "PASS":
    print(json.dumps(VALIDATION_REPORT, indent=2)[:6000])


In [ ]:
# ============================================================
# CELL 7 — Evaluation and interactive inspection
# ============================================================

def evaluate_mapping() -> Dict[str, Any]:
    statuses = Counter(
        row["mapping_status"] for row in ALL_MAPPINGS
    )
    exact_paths = [
        path
        for row in ALL_MAPPINGS
        for path in row["selected_evidence_paths"]
    ]

    flagged = {
        "low_confidence_coverage": [
            row["requirement_id"]
            for row in ALL_MAPPINGS
            if row["mapping_status"] in {
                "covered", "partially_covered"
            }
            and row["mapping_confidence"] < 0.65
        ],
        "large_evidence_bundles": [
            row["requirement_id"]
            for row in ALL_MAPPINGS
            if len(row["selected_evidence_paths"]) > 20
        ],
        "partial_without_declared_missing_information": [
            row["requirement_id"]
            for row in ALL_MAPPINGS
            if row["mapping_status"] == "partially_covered"
            and not row["missing_information"]
        ],
        "coverage_using_only_cross_section_evidence": [
            row["requirement_id"]
            for row in ALL_MAPPINGS
            if row["mapping_status"] in {
                "covered", "partially_covered"
            }
            and row["selected_source_sections"]
            and row["section_key"] not in row["selected_source_sections"]
        ],
    }

    return {
        "status_mix": dict(statuses),
        "exact_path_count": len(exact_paths),
        "unique_exact_path_count": len(set(exact_paths)),
        "evidence_record_count": len(EVIDENCE_STORE),
        "average_paths_per_covered_requirement": round(
            sum(
                len(row["selected_evidence_paths"])
                for row in ALL_MAPPINGS
                if row["mapping_status"] in {
                    "covered", "partially_covered"
                }
            )
            / max(
                sum(
                    row["mapping_status"] in {
                        "covered", "partially_covered"
                    }
                    for row in ALL_MAPPINGS
                ),
                1,
            ),
            2,
        ),
        "validation": VALIDATION_REPORT,
        "flags": flagged,
    }


EVALUATION_REPORT = evaluate_mapping()

evaluation_path = (
    MAPPING_OUTPUT_DIR
    / f"mapping_evaluation_{RESOLVED_BANK_CODE}.json"
)
evaluation_path.write_text(
    json.dumps(EVALUATION_REPORT, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps({
    "status_mix": EVALUATION_REPORT["status_mix"],
    "exact_path_count": EVALUATION_REPORT["exact_path_count"],
    "evidence_record_count": EVALUATION_REPORT["evidence_record_count"],
    "average_paths_per_covered_requirement": (
        EVALUATION_REPORT["average_paths_per_covered_requirement"]
    ),
    "flag_counts": {
        key: len(value)
        for key, value in EVALUATION_REPORT["flags"].items()
    },
}, indent=2))


def inspect_mapping(
    requirement_id: str,
    section: Optional[str] = None,
) -> None:
    sections = (
        [normalize_section_key(section)]
        if section
        else list(MAPPINGS_BY_SECTION)
    )

    for section_key in sections:
        for row in MAPPINGS_BY_SECTION.get(section_key, []):
            if row["requirement_id"] != requirement_id:
                continue

            print("Section:", section_key)
            print("Requirement:", requirement_id)
            print("Status:", row["mapping_status"])
            print("Confidence:", row["mapping_confidence"])
            print("Cross-section:", row["cross_section"])
            print("\nRequirement text:")
            print(row["requirement_text"])

            print("\nCoverage components:")
            for component in row["coverage_components"]:
                print(
                    f"- {'SUPPORTED' if component['supported'] else 'MISSING'}: "
                    f"{component['component']}"
                )
                for evidence in component["schema_evidence"]:
                    print(
                        f"    {evidence['schema_id']} "
                        f"{evidence['source_section']}::"
                        f"{evidence['schema_path']} "
                        f"[{evidence['record_scope']}]"
                    )

            print("\nResolved evidence groups:")
            for group in row["evidence_groups"]:
                print(
                    f"\nRecord: {group['record_key']} "
                    f"({group['qualified_record_path']})"
                )
                print("Context:", group["record_context"])
                for field in group["fields"]:
                    print(
                        f"  - {field['qualified_path']} = "
                        f"{preview(field['value'], 180)}"
                    )

            if row["status_evidence_groups"]:
                print("\nStatus justification:")
                for group in row["status_evidence_groups"]:
                    for field in group["fields"]:
                        print(
                            f"  - {field['qualified_path']} = "
                            f"{preview(field['value'], 180)}"
                        )

            print("\nMissing information:")
            for item in row["missing_information"]:
                print("-", item)

            print("\nDeclared data gaps:")
            for gap in row["declared_data_gaps"]:
                print("-", gap.get("field"), "—", gap.get("reason"))

            print("\nRationale:")
            print(row["mapping_rationale"])
            return

    raise KeyError(f"Requirement not found: {requirement_id}")


# Example:
# inspect_mapping("<requirement_id>", section="<section_key>")
